##### **Gold Layer**
- ##### _Build dimensional model for analytics_
- Column names and types match the published semantic model **05_RPT_Retail_ECommerce_Sales_Analysis**.
- Ends with a data-quality gate: if any check fails the notebook errors and the pipeline does **not** refresh the model.

In [ ]:
-- Create gold schema

CREATE SCHEMA IF NOT EXISTS gold;

In [ ]:
-- Create Region Dimension

CREATE OR REPLACE TABLE gold.dim_region AS
SELECT
    CAST(RegionID AS STRING) AS RegionID,
    RegionName
FROM silver.region;

In [ ]:
-- Create Geography Dimension
-- GeographyKey stays STRING because the published semantic model expects text here
-- (the fact side is BIGINT - align both types in the model when it is next edited).

CREATE OR REPLACE TABLE gold.dim_geography AS
SELECT
    CAST(GeographyID AS STRING) AS GeographyKey,
    CountryName,
    StateCode,
    StateName,
    CityName,
    CAST(PostalCode AS STRING)  AS PostalCode,
    CAST(Latitude AS DOUBLE)    AS Latitude,
    CAST(Longitude AS DOUBLE)   AS Longitude,
    CAST(RegionID AS STRING)    AS RegionID
FROM silver.geography;

In [ ]:
-- Create Customer Dimension (exactly one row per customer)

CREATE OR REPLACE TABLE gold.dim_customer AS
WITH seg AS (
    SELECT CustomerID, CustomerSegmentName,
           ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY CustomerSegmentName) AS rn
    FROM silver.customer_segments
),
addr AS (
    SELECT CustomerID, StateCode, StateName, RegionName,
           ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY StateCode) AS rn
    FROM silver.customer_addresses
),
pref AS (   -- most-used order channel per customer; ties go to the lowest ChannelID
    SELECT CustomerKey,
           CustomerChannelKey AS PreferredChannelKey,
           ROW_NUMBER() OVER (PARTITION BY CustomerKey ORDER BY COUNT(*) DESC, CustomerChannelKey) AS rn
    FROM silver.orders
    GROUP BY CustomerKey, CustomerChannelKey
)
SELECT
    CAST(REGEXP_REPLACE(c.CustomerID, '[^0-9]', '') AS BIGINT) AS CustomerKey,
    c.CustomerID,
    c.CustomerName,
    c.Gender,
    CAST(c.Age AS BIGINT)            AS Age,
    c.AgeBand,
    CAST(c.SignupDate AS TIMESTAMP)  AS SignupDate,
    c.CustomerType,
    c.LoyaltyTier,
    c.AcquisitionSource,
    p.PreferredChannelKey,
    a.StateCode   AS HomeStateCode,
    a.StateName   AS HomeStateName,
    a.RegionName  AS HomeRegion,
    s.CustomerSegmentName
FROM silver.customers c
LEFT JOIN seg  s ON s.CustomerID = c.CustomerID AND s.rn = 1
LEFT JOIN addr a ON a.CustomerID = c.CustomerID AND a.rn = 1
LEFT JOIN pref p ON p.CustomerKey = CAST(REGEXP_REPLACE(c.CustomerID, '[^0-9]', '') AS BIGINT) AND p.rn = 1;

In [ ]:
-- Create Product Dimension

CREATE OR REPLACE TABLE gold.dim_product AS
SELECT
    CAST(REGEXP_REPLACE(ProductID, '[^0-9]', '') AS BIGINT) AS ProductKey,
    ProductID,
    SKU,
    ProductName,
    Brand,
    CAST(DENSE_RANK() OVER (ORDER BY CategoryName, SubcategoryName) AS BIGINT) AS CategoryKey,
    CategoryName,
    SubcategoryName,
    Department,
    ProductType,
    Color,
    Size,
    Season,
    CAST(UnitCostUSD AS DOUBLE)      AS UnitCostUSD,
    CAST(StandardPriceUSD AS DOUBLE) AS StandardPriceUSD,
    CAST(IsActive AS BIGINT)         AS IsActive
FROM silver.products;

In [ ]:
-- Create Sales Fact Table

CREATE OR REPLACE TABLE gold.fact_sales_order_item AS
SELECT
    oi.OrderID,
    oi.OrderItemID,
    TO_DATE(CAST(o.OrderDateKey AS STRING), 'yyyyMMdd') AS OrderDate,
    o.CustomerKey,
    oi.ProductKey,
    o.GeographyKey,
    oi.Quantity,
    oi.UnitPriceUSD,
    oi.GrossSalesUSD,
    oi.DiscountUSD,
    oi.ReturnAmountUSD,
    oi.NetRevenueUSD,
    oi.ProductCostUSD,
    oi.GrossProfitUSD,
    oi.IsReturned,
    oi.IsCancelled,
    oi.IsDelivered,
    oi.IsRepeatCustomer,
    oi.CustomerOrderSequence,
    os.OrderStatusName AS OrderStatus,
    ch.ChannelName     AS CustomerChannel
FROM silver.order_items oi
LEFT JOIN silver.orders o          ON oi.OrderID = o.OrderID
LEFT JOIN silver.order_statuses os ON o.OrderStatusKey = os.OrderStatusID
LEFT JOIN silver.channels ch       ON o.CustomerChannelKey = ch.ChannelID;

In [ ]:
# Data-quality gate: every check returns the number of problem rows (0 = pass)

CHECKS = {
    "fact table is empty":                  "SELECT CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END FROM gold.fact_sales_order_item",
    "fact rows != silver order_items rows": "SELECT ABS((SELECT COUNT(*) FROM gold.fact_sales_order_item) - (SELECT COUNT(*) FROM silver.order_items))",
    "duplicate CustomerKey":                "SELECT COUNT(*) - COUNT(DISTINCT CustomerKey) FROM gold.dim_customer",
    "duplicate ProductKey":                 "SELECT COUNT(*) - COUNT(DISTINCT ProductKey) FROM gold.dim_product",
    "duplicate GeographyKey":               "SELECT COUNT(*) - COUNT(DISTINCT GeographyKey) FROM gold.dim_geography",
    "duplicate RegionID":                   "SELECT COUNT(*) - COUNT(DISTINCT RegionID) FROM gold.dim_region",
    "fact rows without OrderDate":          "SELECT COUNT(*) FROM gold.fact_sales_order_item WHERE OrderDate IS NULL",
    "fact CustomerKey not in dim_customer": "SELECT COUNT(*) FROM gold.fact_sales_order_item f LEFT ANTI JOIN gold.dim_customer d ON f.CustomerKey = d.CustomerKey",
    "fact ProductKey not in dim_product":   "SELECT COUNT(*) FROM gold.fact_sales_order_item f LEFT ANTI JOIN gold.dim_product d ON f.ProductKey = d.ProductKey",
    "fact GeographyKey not in dim_geography": "SELECT COUNT(*) FROM gold.fact_sales_order_item f LEFT ANTI JOIN gold.dim_geography d ON CAST(f.GeographyKey AS STRING) = d.GeographyKey",
}

results = {name: spark.sql(q).first()[0] for name, q in CHECKS.items()}
for name, bad in results.items():
    print(f"{'PASS' if bad == 0 else 'FAIL'}  {name}: {bad}")

failed = {name: bad for name, bad in results.items() if bad != 0}
if failed:
    raise ValueError(f"Gold data-quality gate failed: {failed}")

In [ ]:
# Row counts per gold table (small summary instead of displaying the whole fact table)
display(spark.sql('''
    SELECT 'dim_region' AS table_name, COUNT(*) AS row_count FROM gold.dim_region
    UNION ALL SELECT 'dim_geography', COUNT(*) FROM gold.dim_geography
    UNION ALL SELECT 'dim_customer', COUNT(*) FROM gold.dim_customer
    UNION ALL SELECT 'dim_product', COUNT(*) FROM gold.dim_product
    UNION ALL SELECT 'fact_sales_order_item', COUNT(*) FROM gold.fact_sales_order_item
'''))

In [ ]:
# Sync the lakehouse SQL analytics endpoint so the semantic model refresh sees the new gold tables straight away
import sempy.fabric as fabric

lakehouse_id = fabric.get_lakehouse_id()
try:
    if hasattr(fabric, "refresh_sql_endpoint_metadata"):
        display(fabric.refresh_sql_endpoint_metadata(warehouse=lakehouse_id, warehouse_type="Lakehouse"))
    else:
        # older semantic-link without the helper: call the REST API directly
        client = fabric.FabricRestClient()
        ws = fabric.get_workspace_id()
        lh = client.get(f"/v1/workspaces/{ws}/lakehouses/{lakehouse_id}").json()
        endpoint_id = lh["properties"]["sqlEndpointProperties"]["id"]
        client.post(f"/v1/workspaces/{ws}/sqlEndpoints/{endpoint_id}/refreshMetadata", json={})
        print("SQL endpoint metadata refresh requested")
except Exception as e:
    # Not fatal: the endpoint also syncs on its own within about a minute, and the refresh step retries
    print(f"WARNING: SQL endpoint metadata sync failed: {e}")